# Simple LLM Pipeline (Minimal GPT from Scratch)
This notebook shows the **simplest end‑to‑end pipeline** for a tiny LLM:
1. Load text
2. Tokenize
3. Build embeddings
4. Implement attention
5. Build a tiny GPT model
6. Train on next‑token prediction
7. Generate text

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


## 1. Load Simple Training Text

In [2]:
text = """
Machine learning is fun.
Deep learning builds powerful models.
Language models learn patterns in text.
"""


## 2. Tokenization (character level for simplicity)

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(vocab_size)


26


## 3. Create Training Batches

In [4]:
block_size = 8

def get_batch():
    ix = torch.randint(len(data)-block_size, (4,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y


## 4. Self‑Attention

In [5]:
class SelfAttention(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.key = nn.Linear(n_embd, n_embd)
        self.query = nn.Linear(n_embd, n_embd)
        self.value = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        K = self.key(x)
        Q = self.query(x)
        V = self.value(x)

        scores = Q @ K.transpose(-2,-1) / math.sqrt(x.size(-1))
        weights = F.softmax(scores, dim=-1)

        return weights @ V


## 5. Transformer Block

In [6]:
class Block(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.attn = SelfAttention(n_embd)
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd)
        )

    def forward(self, x):
        x = x + self.attn(x)
        x = x + self.ff(x)
        return x


## 6. Tiny GPT Model

In [7]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=32):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)

        self.block = Block(n_embd)
        self.ln = nn.LayerNorm(n_embd)

        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B,T = idx.shape

        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(T))

        x = tok + pos

        x = self.block(x)

        x = self.ln(x)

        logits = self.head(x)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


## 7. Train Model

In [8]:
model = TinyGPT(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# from range 500 to 5000, mean train longer
for step in range(5000):

    xb, yb = get_batch()

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())


0 3.6044938564300537
100 2.204723834991455
200 1.1168608665466309
300 0.3633071482181549
400 0.4410598576068878
500 0.36169692873954773
600 0.2582630217075348
700 0.06384025514125824
800 0.0856693685054779
900 0.05668029934167862
1000 0.0805584043264389
1100 0.03175493702292442
1200 0.017938561737537384
1300 0.011351406574249268
1400 0.005677587818354368
1500 0.01212711725383997
1600 0.007405479438602924
1700 0.004284349735826254
1800 0.0073207831010222435
1900 0.004458160605281591
2000 0.004236232023686171
2100 0.006055766250938177
2200 0.09133589267730713
2300 0.00428406335413456
2400 0.005300749558955431
2500 0.0031988220289349556
2600 0.0020083102863281965
2700 0.0017609003698453307
2800 0.0015803946880623698
2900 0.011861794628202915
3000 0.001127802417613566
3100 0.024445604532957077
3200 0.001193371950648725
3300 0.0013554898323491216
3400 0.0013512392761185765
3500 0.062039270997047424
3600 0.06130252033472061
3700 0.0010755239054560661
3800 0.0007994947372935712
3900 0.0267200

## 8. Generate Text

In [9]:
def generate(model, start, length=100):

    idx = torch.tensor([encode(start)], dtype=torch.long)

    for _ in range(length):

        idx_cond = idx[:,-block_size:]

        logits,_ = model(idx_cond)

        logits = logits[:,-1,:]

        probs = F.softmax(logits, dim=-1)

        next_id = torch.multinomial(probs,1)

        idx = torch.cat((idx,next_id),dim=1)

    return decode(idx[0].tolist())


print(generate(model, "Machine"))


Machineaningexingex.
Deep learning is fun.
Deep learning is fun.
Deep learning is fun.
Deep learning is fun
